In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, r2_score
from scipy.stats import loguniform
# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_parquet('../data/clean_data/clean_data.parquet')

In [3]:
df.drop(['age', 'partner', 'country', 'state', 'total_population', 'number_of_dependents', 'churn_score', 'churn_value', 'referred_a_friend'], axis=1, inplace=True)

In [4]:
df.to_parquet('../data/model_data/model_data.parquet')

In [5]:
x = df.drop('churn_label', axis=1)
y = df['churn_label']

In [6]:
result = {}

In [7]:
def decisiontree(x,y):
    decision__tree_params = {
        
        'max_depth':[None,3,5,10,20],
        'min_samples_split':[2,3,5,7,10],
        'min_samples_leaf':[1,2,3,4,5],
        'criterion':['gini', 'entropy'],
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    decision_model = DecisionTreeClassifier(random_state=42)
    grid_decision_model = GridSearchCV(estimator=decision_model, param_grid=decision__tree_params, cv=5, scoring='accuracy',verbose=1)
    grid_decision_model.fit(x_train, y_train)
    y_pred = grid_decision_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'decision_tree':{'accuracy':acc, 'best_params':grid_decision_model.best_params_}})
    return result['decision_tree']

In [8]:
# result = decisiontree(x_train, x_test, y_train, y_test)
# result

In [9]:
def randomforest(x,y):
    randomforest_params = {
        
        'n_estimators':[100, 200, 500, 1000],
        'max_depth':[None, 3, 5, 10, 20],
        'min_samples_split':[2, 3, 5, 7, 10],
        'min_samples_leaf':[1, 2, 3, 4, 5],
        'max_features':['sqrt', 'log2'],
        'bootstrap':[True, False],
        'criterion':['gini', 'entropy']
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    randomforest_model = RandomForestClassifier(random_state=42)
    grid_randomforest_model = GridSearchCV(estimator=randomforest_model, param_grid=randomforest_params, cv=3, scoring='accuracy', verbose=2)
    grid_randomforest_model.fit(x_train, y_train)
    y_pred = grid_randomforest_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'randomforest':{'accuracy':acc, 'best_params':grid_randomforest_model.best_params_}})
    return result['randomforest']

In [10]:
def gradientboosting(x,y):
    gradient_params = {
        
        n_estimators:[1000],
        learning_rate:[0.001, 0.01, 0.5, 0.1],
        max_depth:[None, 3, 5, 7],
        min_samples_split:[2, 3, 5, 7, 10],
        min_samples_leaf:[1, 2, 3, 4, 5],
        subsample:[0.6, 0.8, 1]
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    gradient_model = GradientBoostingClassifier(random_state=42)
    grid_gradient_model = GridSearchCV(estimator=gradient_model, param_grid=gradient_params, cv=5, scoring='accuracy', verbose=1)
    grid_gradient_model.fit(x_train, y_train)
    y_pred = grid_gradient_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'gradient_boosting':{'accuracy':acc, 'best_params':grid_gradient_model.best_params_}})
    return result['gradient_boosting']

In [11]:
def xgboost(x,y):
    xgb_params = {
        
    'n_estimators': [1000],               # Set high for early stopping
    'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
    'max_depth': [None, 3, 5, 7, 10],              # Tree complexity
    'subsample': [0.8, 1.0],             # Rows per tree
    'colsample_bytree': [0.8, 1.0],      # Columns per tree
    'grow_policy': ['depthwise', 'lossguide'],
    # Regularization
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0.1, 1, 10],
    'gamma': [0, 0.1, 1],
    # unbalanced
    'min_child_weight': [1, 5, 10],
    # 'eval_metric': ['logloss', 'aucpr']  # Handle class imbalance
        
}
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
    x_temp, x_val, y_temp, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    unbalanced_weight = round(y.value_counts().sort_values()[0]/y.value_counts().sort_values()[1])
    xgb_model = xgb.XGBClassifier(early_stopping_rounds=10, eval_metric='aucpr', scale_pos_weight=unbalanced_weight)
    grid_xgb_model = GridSearchCV(estimator=xgb_model, param_grid=xgb_params, cv=5, scoring='accuracy', verbose=1)
    grid_xgb_model.fit(x_temp, y_temp, eval_set=[(x_val, y_val)])
    y_pred = grid_xgb_model.predit(y_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'xgboost':{'accuracy':acc, 'best_params':grid_xgb_model.best_params_}})
    return result['xgboost']

In [12]:
# random_sample = df.sample(frac=0.2)
# y.value_counts()

In [13]:
# x = random_sample.drop('churn_label',axis=1)
# y = random_sample['churn_label']
# xgboost(x,y)

In [14]:
def logistic(x,y):
    lr_params = {
        
        'C':loguniform(1e-4, 100),
        'penalty':['l1', 'l2'],
        'solver':['liblinear', 'saga'],
        'max_iter':[100, 200, 500, 1000],
        'class_weight':[None, {0:1, 1:2}, {0:1, 1:3},'balanced'],
        'dual':[True, False]
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    lr_model = LogisticRegression(random_state=42)
    grid_lr_model = RandomizedSearchCV(estimator=lr_model, param_distributions=lr_params, cv=5, n_iter=100, random_state=42, scoring='accuracy', verbose=1)
    grid_lr_model.fit(x_train, y_train)
    y_pred = grid_lr_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'lr_model':{'accuracy':acc, 'best_params':grid_lr_model.best_params_}})
    return result['lr_model']

In [15]:
def knn(x,y):
    knn_params = {
        
        'n_neighbors':[i+1 for i in range(10)],
        'weights':['uniform', 'distance'],
        'p':[1, 2]
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    knn_model = KNeighborsClassifier()
    grid_knn_model = GridSearchCV(estimator=knn_model, param_grid=knn_params, cv=5, scoring='accuracy', verbose=1)
    grid_knn_model.fit(x_train, y_train)
    y_pred = grid_knn_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'knn':{'accuracy':acc, 'best_params':grid_knn_model.best_params_}})
    return result['knn']

In [16]:
def gaussianNB(x,y):
    naive_params = {
        
        'var_smoothing':[1e-10, 1e-9, 1e-8, 1e-7, 1e-6]
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    naive_model = GaussianNB()
    naive_grid = GridSearchCV(estimator=naive_model, param_grid=naive_params, cv=5, scoring='accuracy', verbose=1)
    naive_grid.fit(x_train, y_train)
    y_pred = naive_grid.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'naive_bayes':{'accuracy':acc, 'best_params':naive_grid.best_params_}})
    return result['naive_bayes']

In [17]:
y.value_counts()

churn_label
0    5153
1    1864
Name: count, dtype: int64

In [18]:
def linearsvc(x,y):
    svm_params = {
        
        'C':loguniform(1e-4, 100),
        'penalty':['l1', 'l2'],
        'dual':[False, True],
        'loss':['hinge', 'squared_hinge'],
        'class_weight':[None, {0:1, 1:2}, {0:1, 1:3},'balanced'],
        
    }
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)
    svm_model = LinearSVC(max_iter=10000, random_state=42)
    grid_svm = RandomizedSearchCV(estimator=svm_model, param_distributions=svm_params, cv=5, n_iter=50, random_state=42, scoring='accuracy', verbose=1)
    grid_svm.fit(x_train, y_train)
    y_pred = grid_svm.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'svm':{'accuracy':acc, 'best_params':grid_svm.best_params_}})
    return result['svm']

In [19]:
decisiontree(x,y)
# randomforest(x,y)
# gradientboosting(x,y)
# xgboost(x,y)
# logistic(x,y)
gaussianNB(x,y)
knn(x,y)
linearsvc(x,y)

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Fitting 5 folds for each of 50 candidates, totalling 250 fits


/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase

{'accuracy': 0.9430199430199431,
 'best_params': {'C': 0.5492736444508451,
  'class_weight': {0: 1, 1: 2},
  'dual': False,
  'loss': 'squared_hinge',
  'penalty': 'l1'}}

In [20]:
result

{'decision_tree': {'accuracy': 0.9441595441595442,
  'best_params': {'criterion': 'entropy',
   'max_depth': 10,
   'min_samples_leaf': 5,
   'min_samples_split': 2}},
 'naive_bayes': {'accuracy': 0.8774928774928775,
  'best_params': {'var_smoothing': 1e-08}},
 'knn': {'accuracy': 0.7589743589743589,
  'best_params': {'n_neighbors': 8, 'p': 1, 'weights': 'uniform'}},
 'svm': {'accuracy': 0.9430199430199431,
  'best_params': {'C': 0.5492736444508451,
   'class_weight': {0: 1, 1: 2},
   'dual': False,
   'loss': 'squared_hinge',
   'penalty': 'l1'}}}

In [27]:
response = str(input()) # done

 done


In [28]:
if response=='done':
    import json
    with open('result.json', 'w') as file:
        file.write(json.dumps(result))
    result_df = pd.DataFrame(result)
else:
    print('give response')

In [29]:
result_dft = result_df.T
result_dft

,accuracy,best_params
decision_tree,0.94416,"{'criterion': 'entropy', 'max_depth': 10, 'min..."
naive_bayes,0.877493,{'var_smoothing': 1e-08}
knn,0.758974,"{'n_neighbors': 8, 'p': 1, 'weights': 'uniform'}"
svm,0.94302,"{'C': 0.5492736444508451, 'class_weight': {0: ..."


In [30]:
try:
    result_dft.sort_values('accuracy', axis=0, ascending=False, inplace=True)
except KeyError as key:
    print(key, 'Check result!!!, make sure its not empty.')

In [31]:
result_dft

,accuracy,best_params
decision_tree,0.94416,"{'criterion': 'entropy', 'max_depth': 10, 'min..."
svm,0.94302,"{'C': 0.5492736444508451, 'class_weight': {0: ..."
naive_bayes,0.877493,{'var_smoothing': 1e-08}
knn,0.758974,"{'n_neighbors': 8, 'p': 1, 'weights': 'uniform'}"


In [26]:
result_dft.to_csv('../data/result/model_result.csv')

OSError: Cannot save file into a non-existent directory: '../data/result'